In [37]:
import pandas as pd
import requests
from io import StringIO
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import re
import numpy as np
import plotly.express as px

### ***Coletando os dados de cada jogo***

### LOL

In [38]:
url_lol = 'https://turbosmurfs.gg/article/league-of-legends-player-count-and-statistics'

headers = {
    'User-Agent': 'Mozilla/5.0'
}

response = requests.get(url_lol, headers=headers)

soup = BeautifulSoup(response.text, 'html.parser')

texto = soup.get_text(' ', strip=True)

padrao = r'(20\d{2}|2011|2012|2013|2014|2015|2016|2017|2018|2019)\s+(\d+)\s+Million\s*([+-]?\d+(?:\.\d+)?%)'

dados = re.findall(padrao, texto)

#DataFrame
df_lol = pd.DataFrame(
    dados,
    columns=[
        'Ano',
        'Jogadores_Ativos_Mensais',
        'Crescimento_Percentual'
    ]
)

#Conversão dos tipos
df_lol['Ano'] = df_lol['Ano'].astype(int)

df_lol['Jogadores_Ativos_Mensais'] = (
    df_lol['Jogadores_Ativos_Mensais']
    .astype(float)
)

df_lol['Crescimento_Percentual'] = (
    df_lol['Crescimento_Percentual']
    .str.replace('%', '', regex=False)
    .astype(float)
)

print(df_lol)

#Salvar CSV
df_lol.to_csv(
    'jogadores_lol.csv',
    index=False,
    encoding='utf-8-sig'
)

     Ano  Jogadores_Ativos_Mensais  Crescimento_Percentual
0   2011                      15.0                   200.0
1   2012                      32.0                   113.0
2   2013                      67.0                   110.0
3   2014                      70.0                     4.5
4   2015                      90.0                    28.0
5   2016                     100.0                    11.0
6   2017                      81.0                   -18.9
7   2018                      75.0                    -7.5
8   2019                     116.0                    54.5
9   2020                     137.0                    18.0
10  2021                     149.0                     8.0
11  2022                     152.0                     2.0
12  2023                     152.0                    -0.7
13  2024                     132.0                   -12.5
14  2025                     131.0                    -0.7


In [39]:
df_lol.info()

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 3 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Ano                       15 non-null     int64  
 1   Jogadores_Ativos_Mensais  15 non-null     float64
 2   Crescimento_Percentual    15 non-null     float64
dtypes: float64(2), int64(1)
memory usage: 492.0 bytes


### Dota 2

In [40]:
url_dota2 = 'https://steamplayercount.com/app/570'

headers = {
    'User-Agent': 'Mozilla/5.0'
}

response = requests.get(url_dota2, headers=headers)

html = StringIO(response.text)

tabelas = pd.read_html(html)

print(f'Foram encontradas {len(tabelas)} tabelas.')

for i, tabela in enumerate(tabelas):
    print(f'\nTabela {i}')
    print(tabela.head())

Foram encontradas 3 tabelas.

Tabela 0
                   0             1              2
0             298726             0        1290038
1  players right now  24-hour peak  all-time peak

Tabela 1
           Month    Peak    Gain % Gain  Min Daily Peak  Avg Daily Peak
0  February 2026  870671  +13161    +2%          743693          788656
1   January 2026  857510  -47739    -5%          705157          771469
2  December 2025  905249   -3840    -1%          687249          800206
3  November 2025  909089  +45253    +5%          699238          798545
4   October 2025  863836  -83252    -9%          716015          767147

Tabela 2
   Games           Games.1  Players Now  7d Peak
0    NaN  Counter-Strike 2      1368101  1590536
1    NaN          Portal 2         1813     2977
2    NaN  Source Filmmaker         1016     1339
3    NaN             Steam          359      368
4    NaN    DOOM + DOOM II          234      390


In [41]:
df_dota2 = pd.DataFrame(tabelas[1])

df_dota2.head(10)

,Month,Peak,Gain,% Gain,Min Daily Peak,Avg Daily Peak
0,February 2026,870671,+13161,+2%,743693,788656
1,January 2026,857510,-47739,-5%,705157,771469
2,December 2025,905249,-3840,-1%,687249,800206
3,November 2025,909089,+45253,+5%,699238,798545
4,October 2025,863836,-83252,-9%,716015,767147
5,September 2025,947088,+125943,+15%,719572,818803
6,August 2025,821145,+159505,+24%,604980,712893
7,July 2025,661640,+8887,+1%,542293,591680
8,June 2025,652753,-10297,-2%,540936,587124
9,May 2025,663050,+26605,+4%,522351,585176


In [42]:
df_dota2 = df_dota2.rename(columns={
    'Month': 'Mês',
    'Peak': 'Pico',
    'Gain': 'Ganho',
    '% Gain': 'Ganho%',
    'Min Daily Peak': 'Pico_Mínimo_Diário',
    'Avg Daily Peak': 'Pico_Médio_Diário'
})

df_dota2.head()

,Mês,Pico,Ganho,Ganho%,Pico_Mínimo_Diário,Pico_Médio_Diário
0,February 2026,870671,+13161,+2%,743693,788656
1,January 2026,857510,-47739,-5%,705157,771469
2,December 2025,905249,-3840,-1%,687249,800206
3,November 2025,909089,+45253,+5%,699238,798545
4,October 2025,863836,-83252,-9%,716015,767147


In [43]:
df_dota2.to_csv(
    'jogadores_dota2.csv',
    index=False,
    encoding='utf-8-sig'
)

In [44]:
df_dota2.info()

<class 'pandas.DataFrame'>
RangeIndex: 174 entries, 0 to 173
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Mês                 174 non-null    str  
 1   Pico                174 non-null    int64
 2   Ganho               174 non-null    str  
 3   Ganho%              174 non-null    str  
 4   Pico_Mínimo_Diário  174 non-null    int64
 5   Pico_Médio_Diário   174 non-null    int64
dtypes: int64(3), str(3)
memory usage: 8.3 KB


In [45]:
df_dota2['Mês'] = pd.to_datetime(df_dota2['Mês'])

df_dota2['Ano'] = df_dota2['Mês'].dt.year

df_dota2_anual = (
    df_dota2
    .groupby('Ano', as_index=False)['Pico']
    .mean()
)

df_dota2_anual['Crescimento_Percentual'] = (
    df_dota2_anual['Pico']
    .pct_change() * 100
).round(0).fillna(0)

df_dota2_anual.head()

C:\Users\drrec\AppData\Local\Temp\ipykernel_612\533179655.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_dota2['Mês'] = pd.to_datetime(df_dota2['Mês'])


,Ano,Pico,Crescimento_Percentual
0,2011,8351.750000,0.0
1,2012,93936.083333,1025.0
2,2013,444866.833333,374.0
3,2014,820927.583333,85.0
4,2015,985203.416667,20.0


## Função para puxar os dados do site Steamplayercount

In [46]:
def carregar_dados(url, nome_csv=None):

    headers = {
        'User-Agent': 'Mozilla/5.0'
    }

    response = requests.get(url, headers=headers)

    html = StringIO(response.text)

    tabelas = pd.read_html(html)

    df = tabelas[1].copy()

    df = df.rename(columns={
        'Month': 'Mês',
        'Peak': 'Pico',
        'Gain': 'Ganho',
        '% Gain': 'Ganho%',
        'Min Daily Peak': 'Pico_Mínimo_Diário',
        'Avg Daily Peak': 'Pico_Médio_Diário'
    })

    if nome_csv is not None:
        df.to_csv(
            nome_csv,
            index=False,
            encoding='utf-8-sig'
        )

    df['Mês'] = pd.to_datetime(df['Mês'])

    df['Ano'] = df['Mês'].dt.year

    df_anual = (
        df
        .groupby('Ano', as_index=False)['Pico']
        .mean()
    )

    df_anual['Crescimento_Percentual'] = (
        df_anual['Pico']
        .pct_change() * 100
    ).round(0).fillna(0)

    return df, df_anual

### Smite

In [47]:
url_smite = 'https://steamplayercount.com/app/386360'
df_smite, df_smite_anual = carregar_dados(
    url_smite,
    'jogadores_smite.csv'
)
df_smite.head(10)

C:\Users\drrec\AppData\Local\Temp\ipykernel_612\70100658.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Mês'] = pd.to_datetime(df['Mês'])


,Mês,Pico,Ganho,Ganho%,Pico_Mínimo_Diário,Pico_Médio_Diário,Ano
0,2026-02-01,2238,-351,-14%,1703,1884,2026
1,2026-01-01,2589,+287,+13%,1912,2148,2026
2,2025-12-01,2302,-4,0%,1703,1943,2025
3,2025-11-01,2306,+44,+2%,1763,1948,2025
4,2025-10-01,2262,-36,-2%,1655,1886,2025
5,2025-09-01,2298,-238,-9%,1729,2003,2025
6,2025-08-01,2536,-1,0%,1903,2160,2025
7,2025-07-01,2537,-170,-6%,1931,2164,2025
8,2025-06-01,2707,-249,-9%,1987,2286,2025
9,2025-05-01,2956,-129,-4%,2119,2416,2025


#### Juntando os DataFrames com exceção do DataFrame do lol

In [48]:
df_dota2 = pd.read_csv('jogadores_dota2.csv')
df_smite = pd.read_csv('jogadores_smite.csv')

In [49]:
df_dota2['Jogo'] = 'Dota 2'
df_smite['Jogo'] = 'Smite'

df_mobas = pd.concat(
    [df_dota2, df_smite],
    ignore_index=True
)
df_mobas.head()

,Mês,Pico,Ganho,Ganho%,Pico_Mínimo_Diário,Pico_Médio_Diário,Jogo
0,February 2026,870671,+13161,+2%,743693,788656,Dota 2
1,January 2026,857510,-47739,-5%,705157,771469,Dota 2
2,December 2025,905249,-3840,-1%,687249,800206,Dota 2
3,November 2025,909089,+45253,+5%,699238,798545,Dota 2
4,October 2025,863836,-83252,-9%,716015,767147,Dota 2


In [50]:
df_mobas.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Mês                 300 non-null    str  
 1   Pico                300 non-null    int64
 2   Ganho               300 non-null    str  
 3   Ganho%              300 non-null    str  
 4   Pico_Mínimo_Diário  300 non-null    int64
 5   Pico_Médio_Diário   300 non-null    int64
 6   Jogo                300 non-null    str  
dtypes: int64(3), str(4)
memory usage: 16.5 KB


In [51]:
df_mobas['Jogo'].value_counts()

Jogo
Dota 2    174
Smite     126
Name: count, dtype: int64

### Análises

In [52]:
fig = px.line(
    df_lol,
    x='Ano',
    y='Jogadores_Ativos_Mensais',
    markers=True
)

fig.update_layout(
    title='Evolução dos jogadores ativos de League of Legends',
    xaxis_title='Ano',
    yaxis_title='Jogadores ativos (milhões)',
    template='plotly_white',
    hovermode='x unified'
)

fig.update_traces(
    line=dict(width=3),
    marker=dict(size=8)
)

fig.show()

In [53]:
df_mobas['Mês'] = pd.to_datetime(df_mobas['Mês'])
df_mobas['Ano'] = df_mobas['Mês'].dt.year
df_mobas_anual = (
    df_mobas
    .groupby(['Jogo', 'Ano'], as_index=False)['Pico']
    .mean()
)
df_mobas_anual['Crescimento_Percentual'] = (
    df_mobas_anual
    .groupby('Jogo')['Pico']
    .pct_change()
    .mul(100)
    .round(1)
    .fillna(0)
)

C:\Users\drrec\AppData\Local\Temp\ipykernel_612\2572781885.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_mobas['Mês'] = pd.to_datetime(df_mobas['Mês'])


In [57]:
px.line(
    df_mobas_anual,
    x='Ano',
    y='Pico',
    color='Jogo',
    markers=True
)


#### Convertendo a coluna mês para ano

In [64]:
#Dota 2
df_dota2['Mês'] = pd.to_datetime(df_dota2['Mês'])

df_dota2['Ano'] = df_dota2['Mês'].dt.year

df_dota2_anual = (
    df_dota2
    .groupby('Ano', as_index=False)['Pico']
    .mean()
)

df_dota2_anual['Jogo'] = 'Dota 2'

#Smite
df_smite['Mês'] = pd.to_datetime(df_smite['Mês'])

df_smite['Ano'] = df_smite['Mês'].dt.year

df_smite_anual = (
    df_smite
    .groupby('Ano', as_index=False)['Pico']
    .mean()
)

df_smite_anual['Jogo'] = 'SMITE'

#Unindo os DataFrames
df_mobas = pd.concat(
    [df_dota2_anual, df_smite_anual],
    ignore_index=True
)

C:\Users\drrec\AppData\Local\Temp\ipykernel_612\59243889.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_dota2['Mês'] = pd.to_datetime(df_dota2['Mês'])
C:\Users\drrec\AppData\Local\Temp\ipykernel_612\59243889.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_smite['Mês'] = pd.to_datetime(df_smite['Mês'])


In [65]:
fig = px.line(
    df_mobas,
    x='Ano',
    y='Pico',
    color='Jogo',
    markers=True
)

fig.update_layout(
    title='Evolução do pico médio de jogadores dos MOBAs',
    xaxis_title='Ano',
    yaxis_title='Pico médio de jogadores',
    template='plotly_white',
    hovermode='x unified'
)

fig.update_traces(
    line=dict(width=3),
    marker=dict(size=8)
)

fig.show()

### Convertendo tudo para base 100 para comparar com lol

Os dados utilizados provêm de fontes diferentes. Para League of Legends foram utilizados jogadores ativos mensais, enquanto para Dota 2 e SMITE foram utilizados picos médios de jogadores simultâneos. Como essas métricas possuem escalas diferentes, a comparação entre os jogos será realizada por meio do Índice Base 100, permitindo analisar a evolução relativa de cada título ao longo do tempo, e não seus valores absolutos.

In [66]:
#Lol
df_lol = df_lol.rename(
    columns={'Jogadores_Ativos_Mensais': 'Jogadores'}
)

df_lol['Jogo'] = 'League of Legends'

# Manter apenas as colunas necessárias
df_lol = df_lol[['Ano', 'Jogadores', 'Jogo']]

#DataFrame mobas
df_mobas = df_mobas.rename(
    columns={'Pico': 'Jogadores'}
)

df_mobas = df_mobas[['Ano', 'Jogadores', 'Jogo']]

#Unindo os DataFrames
df_todos = pd.concat(
    [df_lol, df_mobas],
    ignore_index=True
)
df_todos = df_todos.sort_values(['Jogo', 'Ano'])

#Converção para base 100
df_todos['Base_100'] = (
    df_todos
    .groupby('Jogo')['Jogadores']
    .transform(lambda x: (x / x.iloc[0]) * 100)
)

In [67]:
fig = px.line(
    df_todos,
    x='Ano',
    y='Base_100',
    color='Jogo',
    markers=True
)

fig.update_layout(
    title='Comparação dos MOBAs (Índice Base 100)',
    xaxis_title='Ano',
    yaxis_title='Índice Base 100',
    template='plotly_white',
    hovermode='x unified'
)

fig.update_traces(
    line=dict(width=3),
    marker=dict(size=8)
)

fig.show()